# Day 1 — Naive RAG from Primitives

**Goal:** build an end-to-end RAG pipeline with *no framework magic*. Every
line is yours. By the end of today, you can point to every step and explain
what it does.

We'll use:
- **Plain Python** for ingestion and chunking
- **sentence-transformers** (free, local) AND **OpenAI embeddings** (paid) so we can compare
- **Chroma** as the vector store
- **Anthropic Claude** for generation

No LangChain. No LlamaIndex. You'll meet them later — first, you need to know
what they're abstracting.

## 1. Setup

This notebook intentionally builds a minimal RAG system from primitive pieces so you can see every moving part instead of hiding the logic behind a framework.

**What to focus on**
- A RAG pipeline is really just: load documents, split them, embed them, retrieve the nearest chunks, then ask an LLM to answer from that context.
- Each step introduces tradeoffs. Simpler code is easier to understand, but often worse for quality, cost, or latency.
- We keep the implementation explicit so later improvements in Days 2-4 feel motivated rather than magical.


In [3]:
# Load shared configuration and helpers once so later cells can stay focused on RAG logic.
# We also instantiate the LLM and vector DB up front to fail fast if environment setup is broken.
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path
from utils import (
    get_llm_client_grok, get_chroma_client,
    Chunk, make_chunk_id, add_chunks,
)

client, generate = get_llm_client_grok()
chroma = get_chroma_client("./chroma_db")
print("Setup OK")


Setup OK


## 2. Load the clean corpus

Day 0 downloaded 5 PEPs to `./datasets/day1/`. Read them as-is.

In production, corpus loading is where data quality issues begin. File naming, encoding, and source tracking all matter because retrieval quality depends on clean inputs.

**Why this section matters**
- We preserve the original filename as metadata so answers can cite sources later.
- Even before chunking, it is useful to inspect document sizes because very short and very long files behave differently during splitting and embedding.


In [4]:
# Keep both raw text and source metadata.
# Source tracking is essential later for citations, debugging retrieval misses, and evaluation.
docs = []
for path in sorted(Path("./datasets/day1").glob("*.md")):
    text = path.read_text()
    docs.append({"source": path.name, "text": text})
    print(f"  {path.name}: {len(text):,} chars")

print(f"\nLoaded {len(docs)} documents.")


  pep-0008.md: 43,371 chars
  pep-0020.md: 1,567 chars
  pep-0257.md: 9,707 chars
  pep-0328.md: 9,218 chars
  pep-0484.md: 81,727 chars

Loaded 5 documents.


## 3. Chunking — the naive version

Split each doc into ~500-token chunks with ~50-token overlap. We'll use a
simple character-based splitter: 1 token ≈ 4 chars for English, so 500 tokens
≈ 2000 chars.

**This chunking is bad.** It doesn't respect sentence or paragraph boundaries.
That's the point — we want to see it fail so tomorrow's improvements are
motivated.

Naive chunking is deliberately crude: it slices by character count without respecting paragraphs, headings, or sentence boundaries.

**What to notice**
- It is easy to implement and surprisingly competitive on simple corpora.
- It can split a key idea across chunk boundaries, which hurts retrieval recall.
- Overlap is a cheap way to reduce boundary damage, but it also increases storage and embedding cost.


In [5]:
# This splitter is intentionally naive: fixed-size character windows plus overlap.
# The overlap softens boundary failures when an important idea spans two adjacent chunks.
def naive_split(text: str, chunk_size: int = 2000, overlap: int = 200) -> list[str]:
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i : i + chunk_size])
        i += chunk_size - overlap
    return chunks


all_chunks: list[Chunk] = []
for doc in docs:
    pieces = naive_split(doc["text"])
    for idx, piece in enumerate(pieces):
        all_chunks.append(Chunk(
            id=make_chunk_id(doc["source"], idx, piece),
            text=piece,
            source=doc["source"],
            extra={"chunk_idx": idx, "total_chunks": len(pieces)},
        ))

print(f"Created {len(all_chunks)} chunks from {len(docs)} documents.")
print(f"Sample chunk:\n---\n{all_chunks[0].text[:300]}...\n---")


Created 84 chunks from 5 documents.
Sample chunk:
---
# PEP-0008

Source: https://peps.python.org/pep-0008/

PEP 8 – Style Guide for Python Code
- Author:
- Guido van Rossum <guido at python.org>, Barry Warsaw <barry at python.org>, Alyssa Coghlan <ncoghlan at gmail.com>
- Status:
- Active
- Type:
- Process
- Created:
- 05-Jul-2001
- Post-History:
- 05...
---


## 4. Embed with sentence-transformers (free, local)

MiniLM-L6-v2 is tiny (80 MB), fast on CPU, and good enough for most prototypes.
First call downloads the model.

Embedding turns text into vectors so semantically similar passages land near each other in vector space.

**Reasoning note**
- We start with a local sentence-transformers model because it is free and fast enough for a workshop-sized corpus.
- The goal is not to find the perfect model yet; the goal is to make the retrieval loop tangible.


In [6]:
# Sentence-transformers gives us local embeddings, which keeps the first end-to-end pipeline free and reproducible.
# We embed all chunk texts once, then reuse those vectors for nearest-neighbor search.
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [c.text for c in all_chunks]
st_embeddings = st_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print(f"\nEmbedded {len(texts)} chunks → shape {st_embeddings.shape}")


/Users/kevin/Programming/rag-bootcamp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12286.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00,  3.30it/s]


Embedded 84 chunks → shape (84, 384)


## 5. Store in Chroma (collection 1: local embeddings)

Chroma is the vector database layer. It stores each chunk together with embeddings and metadata so we can run nearest-neighbor search later.

**Important concept**
- The vector database does not improve your data by itself. Good retrieval still depends on chunking strategy, metadata quality, and consistent embeddings.


In [7]:
# Recreate the collection each run so notebook state stays deterministic.
# Passing explicit embeddings keeps the storage layer separate from model choice.
# Reset so re-running this notebook is clean
try:
    chroma.delete_collection("day1_local")
except Exception:
    pass

coll_local = chroma.create_collection(
    name="day1_local",
    # We're providing embeddings explicitly, so tell Chroma not to embed itself
    metadata={"embedding": "all-MiniLM-L6-v2"},
)

add_chunks(
    coll_local,
    all_chunks,
    embeddings=st_embeddings.tolist()
)
print(f"Collection 'day1_local' has {coll_local.count()} chunks.")


Collection 'day1_local' has 84 chunks.


## 6. Embed with OpenAI (paid, stronger)

Only run if you have `OPENAI_API_KEY` set. Comment out if not.

This optional section mirrors the same pipeline with a stronger hosted embedding model.

**Why compare both**
- Better embeddings often improve recall, especially when the query and source use different wording.
- Hosted models usually add cost and dependency risk, so comparison helps you decide whether the gain is worth it.


In [ ]:
# This optional cell shows how to swap in a hosted embedding model without changing the rest of the pipeline.
# The comparison is useful because better embeddings can improve recall even when everything else stays constant.
# import os

# USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

# if USE_OPENAI:
#     from openai import OpenAI
#     oai = OpenAI()

#     # Batch for API efficiency
#     def embed_openai(texts: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
#         # The API handles batching up to a limit; for small corpora just call once
#         resp = oai.embeddings.create(model=model, input=texts)
#         return [d.embedding for d in resp.data]

#     oai_embeddings = embed_openai([c.text for c in all_chunks])
#     print(f"OpenAI embeddings: {len(oai_embeddings)} vectors of dim {len(oai_embeddings[0])}")

#     try:
#         chroma.delete_collection("day1_openai")
#     except Exception:
#         pass
#     coll_openai = chroma.create_collection(
#         name="day1_openai",
#         metadata={"embedding": "text-embedding-3-small"},
#     )
#     add_chunks(coll_openai, all_chunks, embeddings=oai_embeddings)
#     print(f"Collection 'day1_openai' has {coll_openai.count()} chunks.")
# else:
#     print("Skipping OpenAI embeddings (no API key). Day 1 works fine with only local.")
#     coll_openai = None


## 7. The retrieve function

Take a query, embed it, hit Chroma, return the top-k chunks.

Retrieval is the bridge between your corpus and generation. Its job is not to answer the question; its job is to surface the most relevant evidence.

**Mental model**
- Embeddings power semantic matching.
- The collection returns candidate chunks plus metadata.
- Everything downstream is only as good as the retrieved evidence.


In [8]:
# Retrieval only finds candidate evidence; it does not answer the question yet.
# We normalize the Chroma response into a simpler structure so later cells are easier to read.
def retrieve(query: str, collection, embed_fn, k: int = 4) -> list[dict]:
    q_vec = embed_fn([query])[0]
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return hits


def embed_local(texts: list[str]) -> list[list[float]]:
    return st_model.encode(texts, convert_to_numpy=True).tolist()


# Smoke test
hits = retrieve("what is the zen of python", coll_local, embed_local, k=3)
for h in hits:
    print(f"[{h['distance']:.3f}] {h['metadata']['source']} — {h['text'][:100]}...")


[0.967] pep-0020.md — # PEP-0020

Source: https://peps.python.org/pep-0020/

PEP 20 – The Zen of Python
- Author:
- Tim Pe...
[1.077] pep-0328.md — background, see the following python-dev threads:
Copyright
This document has been placed in the pub...
[1.123] pep-0008.md — es
Copyright
This document has been placed in the public domain.
Source: https://github.com/python/p...


## 8. The generate function

Format retrieved chunks into a prompt and call Claude. Notice we:
- tell Claude to ground in the provided sources
- ask for a short answer
- require it to cite the source file

These are the three hinges of RAG generation. Tune them for your use case.

Generation should be grounded, narrow, and explicit about uncertainty.

**What this section teaches**
- Prompting is doing real engineering work here: we tell the model to stay inside the sources, keep answers concise, and expose which files it used.
- A good RAG prompt reduces hallucinations, but it cannot fully compensate for poor retrieval.


In [9]:
# The generation prompt is strict on purpose.
# In RAG, explicit grounding instructions are one of the easiest ways to reduce hallucinations.
SYSTEM_PROMPT = """You are a precise technical assistant.

You will be given a user question and a set of source excerpts. Answer the \
question using ONLY the information in the sources. If the sources do not \
contain the answer, say so explicitly — do not invent information.

Keep your answer concise (3–5 sentences). At the end, list the source files \
you used on a line starting with "Sources: ".
"""


def format_sources(hits: list[dict]) -> str:
    lines = []
    for i, h in enumerate(hits, 1):
        lines.append(f"[Source {i}: {h['metadata']['source']}]")
        lines.append(h["text"])
        lines.append("")
    return "\n".join(lines)


def rag_answer(question: str, collection, embed_fn, k: int = 4) -> dict:
    hits = retrieve(question, collection, embed_fn, k=k)
    user = f"Sources:\n\n{format_sources(hits)}\n\nQuestion: {question}\n\nAnswer:"
    answer = generate(system=SYSTEM_PROMPT, user=user)
    return {"question": question, "answer": answer, "hits": hits}


## 9. Exercise 1.1 — Ask five questions

Five questions covering the corpus. Note how the retrieved sources change per
question — that's the "R" in RAG working.

These questions are a lightweight sanity check for the whole pipeline.

**How to use them**
- Look for both answer quality and source quality.
- If the answer is wrong, inspect whether retrieval missed the right chunk or generation misused a good chunk.
- That debugging habit becomes central later in the course.


In [10]:
# These sample questions are a quick pipeline smoke test.
# When something fails, inspect both the answer text and the retrieved sources before blaming the LLM.
questions = [
    "What does PEP 8 recommend for the maximum line length?",
    "What is the Zen of Python's rule about readability?",
    "What does PEP 257 say about one-line docstrings?",
    "What are type hints in Python, according to PEP 484?",
    "How should imports be organized per PEP 8?",
]

for q in questions:
    print("=" * 70)
    print(f"Q: {q}")
    r = rag_answer(q, coll_local, embed_local)
    print(f"\nA: {r['answer']}")
    print(f"\n(Retrieved from: {', '.join(h['metadata']['source'] for h in r['hits'])})")
    print()


Q: What does PEP 8 recommend for the maximum line length?

A: PEP 8 recommends a maximum line length of 79 characters. For flowing long blocks of text with fewer structural restrictions, the line length should be limited to 72 characters.

Sources: 
[Source 1: pep-0008.md], 
[Source 2: pep-0008.md], 
[Source 3: pep-0008.md].

(Retrieved from: pep-0008.md, pep-0008.md, pep-0008.md, pep-0257.md)

Q: What is the Zen of Python's rule about readability?

A: According to PEP-0020, the Zen of Python states that "Readability counts." This is mentioned in the excerpt under the title "The Zen of Python".

Sources: 
[Source 1: pep-0020.md], [Source 2: pep-0008.md] is listed but not actually used, but it is actually referenced within the first source.

(Retrieved from: pep-0020.md, pep-0008.md, pep-0257.md, pep-0008.md)

Q: What does PEP 257 say about one-line docstrings?

A: According to PEP 257, one-line docstrings do not have a specific form specified.

However, it is mentioned that a one-line 

## 10. Exercise 1.2 — Conversation memory + the hallucination check

Add a simple memory buffer. Then ask a question the corpus *cannot* answer.
Watch what happens.

Follow-up questions reveal a common weakness in naive RAG: the current query may depend on previous turns.

**Key idea**
- Conversation memory can improve retrieval by rewriting or expanding the current query with recent context.
- This is still a heuristic, not full conversational state management, but it is often enough to improve workshop demos.


In [11]:
# This class adds lightweight conversational memory by expanding the retrieval query with recent turns.
# It is a simple heuristic, but it often helps with follow-up questions like 'what about that one?'
class ChatRAG:
    def __init__(self, collection, embed_fn, k: int = 4, memory_turns: int = 3):
        self.collection = collection
        self.embed_fn = embed_fn
        self.k = k
        self.memory_turns = memory_turns
        self.history: list[tuple[str, str]] = []

    def ask(self, question: str) -> dict:
        # Simple approach: prepend last N turns into the question for retrieval
        # This helps when the user says "what about X?" as a follow-up
        recent = self.history[-self.memory_turns:]
        context_query = " ".join([f"{q} {a}" for q, a in recent] + [question])

        hits = retrieve(context_query, self.collection, self.embed_fn, k=self.k)

        # Include conversation history in the generation prompt
        history_text = "\n".join(f"User: {q}\nAssistant: {a}" for q, a in recent)
        user = (
            f"Prior conversation:\n{history_text}\n\n"
            f"Sources:\n{format_sources(hits)}\n\n"
            f"Current question: {question}\n\nAnswer:"
        )
        answer = generate(system=SYSTEM_PROMPT, user=user)
        self.history.append((question, answer))
        return {"question": question, "answer": answer, "hits": hits}


chat = ChatRAG(coll_local, embed_local)

print("Q1:", "What does PEP 8 say about indentation?")
r1 = chat.ask("What does PEP 8 say about indentation?")
print("A1:", r1["answer"])
print()
print("Q2 (follow-up):", "What about blank lines?")
r2 = chat.ask("What about blank lines?")
print("A2:", r2["answer"])
print()
# The gotcha: ask something not in the corpus
print("Q3 (not in corpus):", "What's the stock price of Nvidia?")
r3 = chat.ask("What's the stock price of Nvidia?")
print("A3:", r3["answer"])


Q1: What does PEP 8 say about indentation?
A1: According to PEP 8, block comments are generally indented to the same level as the code they apply to. 

In addition, inline comments should start with a # and a single space, and should not state the obvious. 

Sources:
[Source 1: pep-0008.md], [Source 3: pep-0008.md]

Q2 (follow-up): What about blank lines?
A2: According to PEP 8, the use of blank lines is not explicitly specified, but it does mention that consistency within a project is more important than general consistency. It is recommended to use your best judgment when deciding when to use a blank line.

Sources: 
[Source 1: pep-0008.md], [Source 2: pep-0008.md], [Source 3: pep-0008.md], [Source 4: pep-0008.md],

Q3 (not in corpus): What's the stock price of Nvidia?
A3: The source excerpts do not contain any information about the stock price of Nvidia.

Sources:
[Source 1: pep-0008.md], [Source 2: pep-0008.md], [Source 3: pep-0008.md], [Source 4: pep-0008.md]


## 11. Compare local vs OpenAI embeddings (if you ran both)

If you have an OpenAI key, rerun Q1 against both collections and compare.
The differences are often subtle on easy questions but big on technical
terminology or multi-hop queries.

Model comparisons are only useful when you hold the rest of the pipeline steady.

**Interpretation tip**
- If one embedding model wins, ask whether it retrieved better evidence, handled paraphrase better, or simply benefited from lucky phrasing in the eval questions.


In [ ]:
# Keep the comparison code optional so the notebook still runs without a hosted embedding API key.
# The main lesson is interface compatibility: if the embed function shape stays stable, swapping models is easy.
# if coll_openai is not None:
#     def embed_openai_fn(texts: list[str]) -> list[list[float]]:
#         return embed_openai(texts)

#     q = "What does PEP 484 say about generic types?"
#     print("LOCAL (MiniLM):")
#     r_local = rag_answer(q, coll_local, embed_local)
#     print(r_local["answer"])
#     print()
#     print("OPENAI (text-embedding-3-small):")
#     r_oai = rag_answer(q, coll_openai, embed_openai_fn)
#     print(r_oai["answer"])


## 12. Reflection

By now you should notice:

1. **Chunks cut mid-sentence.** The naive splitter breaks text at arbitrary
   character offsets. Sometimes the retrieved chunk starts mid-sentence and
   Claude has to guess context. Tomorrow we fix this.

2. **No hierarchy.** We have no idea which section of a PEP a chunk came from.
   For the next 5 days we'll fix this with metadata + structured parsing.

3. **"What's Nvidia's stock price" — did Claude refuse or hallucinate?** If
   it refused, good — the system prompt did its job. If it made something up,
   note it; we'll add stronger guardrails on Day 5.

4. **Run the pipeline on the cursed PDFs tomorrow.** Your Day 1 pipeline
   works on clean markdown. Day 2 will break it.

Save your work and rest up.

Reflection matters because RAG systems are mostly about tradeoffs, not single correct implementations.

**Takeaway questions**
- Where did naive chunking fail?
- When did retrieval fail even though the answer existed?
- Which parts of the system felt robust versus fragile?
